# Multi-column bars in a `Dag`

An eager `agg` dict runs several reducers over one value stream. That is enough
for bar statistics computed from a single series, but real bars are built from
*several* streams at once: open and close come from price, buy and sell volume
come from signed volume, and every column has to agree on where the bars begin
and end.

Inside a graph, the `agg` dict takes the same shape, but its values become lazy
expressions rooted at `Input` nodes rather than functors applied to one array.
Each expression is a per-bar reducer sitting on top of whatever per-tick
transform it needs. One `resample` call then builds a single bar node: one clock,
N labelled columns, bound to data only when the graph is called.

Because all columns hang off that one bar node, they share a single bar clock and
cannot drift relative to each other.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from screamer import First, Last, ExpandingMax, ExpandingMin, ExpandingSum, PosPart, NegPart
from screamer.streams import resample
from screamer.dag import Input, Dag

rng = np.random.default_rng(7)
n = 400
BAR = 40

t_arr     = np.arange(n, dtype=np.int64)
price_arr = 100.0 + np.cumsum(rng.normal(size=n))

# Signed volume: positive is buyer-initiated, negative is seller-initiated.
vol_arr = rng.normal(size=n) * rng.integers(1, 5, size=n)

pd.DataFrame({
    "t":     t_arr[:5],
    "price": price_arr[:5].round(3),
    "vol":   vol_arr[:5].round(2),
}).set_index("t")

## Inputs are named ports, not data

An `Input` is a placeholder for a stream that will be supplied later. Applying a
functor to one does not compute anything: it builds an expression. `First()(price)`
is a node meaning "the first price in each bar", not a number.

This is what lets a reducer sit on top of a per-tick transform. Buy volume is the
running sum of the positive part of signed volume, so the per-tick transform is
`PosPart()(vol)` and the per-bar reducer is `ExpandingSum()`:

```
ExpandingSum()(PosPart()(vol))
```

`PosPart` and `NegPart` are the signed-part decomposition. Every real number
satisfies `x = PosPart(x) - NegPart(x)`, with both parts non-negative: `PosPart`
keeps positive values and zeroes the rest, `NegPart` keeps the magnitude of
negative values and zeroes the rest. Summing each part over a bar splits total
traded volume into buyer- and seller-initiated halves.

In [ ]:
price, vol, t = Input("price"), Input("vol"), Input("t")

# Nothing is computed here. Each value is an expression over the Inputs.
bars = resample(t, every=BAR, agg={
    "open":  First()(price),
    "high":  ExpandingMax()(price),
    "low":   ExpandingMin()(price),
    "close": Last()(price),
    "buy":   ExpandingSum()(PosPart()(vol)),
    "sell":  ExpandingSum()(NegPart()(vol)),
})

print(type(bars).__name__)

## Assembling and calling the graph

`Dag(inputs, outputs)` freezes the expression into a runnable graph. Data binds at
call time: each input is supplied as a `(values, index)` pair, keyed by the name
given to its `Input`. The clock `t` is passed as the first positional argument to
`resample`, so it drives the bar boundaries; `price` and `vol` ride along on the
same clock.

The result is a labelled `Stream` whose `.columns` are the dict keys, in insertion
order.

In [ ]:
dag = Dag([t, price, vol], [bars])

ohlcv = dag(
    t=(t_arr.astype(float), t_arr),
    price=(price_arr, t_arr),
    vol=(vol_arr, t_arr),
)

print("columns:", ohlcv.columns)
pd.DataFrame(
    ohlcv.values.round(2),
    index=pd.Index(ohlcv.index.astype(int), name="bar_open"),
    columns=list(ohlcv.columns),
)

In [ ]:
o, h, l, c = ohlcv["open"], ohlcv["high"], ohlcv["low"], ohlcv["close"]
buy, sell  = ohlcv["buy"], ohlcv["sell"]

fig = plt.figure(figsize=(10, 6))
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

x = np.arange(len(o))
colors = np.where(c >= o, "seagreen", "crimson")
ax1.vlines(x, l, h, color=colors, lw=1)                        # high-low wicks
ax1.bar(x, np.abs(c - o), bottom=np.minimum(o, c),             # open-close bodies
        color=colors, width=0.6)
ax1.set_ylabel("price")
ax1.set_title("OHLC bars with buy and sell volume, one clock")
ax1.tick_params(labelbottom=False)

ax2.bar(x,  buy,  color="seagreen", width=0.6)
ax2.bar(x, -sell, color="crimson",  width=0.6)
ax2.axhline(0, color="k", lw=0.5)
ax2.set_ylabel("volume")
ax2.set_xlabel("bar")

## One clock, no drift

All six columns are produced by a single bar node, so they are finalized at the
same boundaries by construction. The check below rebuilds the price columns
through an ordinary eager `resample` on the same clock and confirms the graph
agrees with it.

The buy and sell columns are checked against the decomposition itself: summing the
positive and negative parts per bar must reproduce them, and their difference is
net order flow.

In [ ]:
# Price columns via the eager path, same clock and bar width.
eager_ohlc = resample(price_arr, t_arr, every=BAR, agg="ohlc")

for name in ("open", "high", "low", "close"):
    np.testing.assert_allclose(ohlcv[name], eager_ohlc[name], rtol=1e-12)

# Buy and sell against the signed-part decomposition.
buy_check  = resample(np.asarray(PosPart()(vol_arr)), t_arr, every=BAR, agg="sum")
sell_check = resample(np.asarray(NegPart()(vol_arr)), t_arr, every=BAR, agg="sum")

np.testing.assert_allclose(ohlcv["buy"],  buy_check.values,  rtol=1e-12)
np.testing.assert_allclose(ohlcv["sell"], sell_check.values, rtol=1e-12)

print("graph columns match the eager path and the signed-part decomposition")

net_flow = ohlcv["buy"] - ohlcv["sell"]
pd.Series(net_flow.round(2), name="net_flow",
          index=pd.Index(ohlcv.index.astype(int), name="bar_open"))

## Extending the bar

The `agg` dict is open-ended. Any expression whose top node is a per-bar reducer
adds a column, and it joins the same clock as the rest. Adding realized volatility
per bar is one more entry, not a second `resample`.

The graph is a value: define it once, and call it with whatever data you bind at
the time.

In [ ]:
from screamer import ExpandingStd

price2, vol2, t2 = Input("price"), Input("vol"), Input("t")
bars2 = resample(t2, every=BAR, agg={
    "close": Last()(price2),
    "vol":   ExpandingStd()(price2),
    "flow":  ExpandingSum()(vol2),
})

out = Dag([t2, price2, vol2], [bars2])(
    t=(t_arr.astype(float), t_arr),
    price=(price_arr, t_arr),
    vol=(vol_arr, t_arr),
)

print("columns:", out.columns)
pd.DataFrame(
    out.values.round(3),
    index=pd.Index(out.index.astype(int), name="bar_open"),
    columns=list(out.columns),
)